# Model Training & Evaluation — Customer Churn Prediction

> **Goal**: Train Logistic Regression, Random Forest, and XGBoost models, compare performance, tune the best model, and generate SHAP explanations.

**Key techniques**: SMOTE for class imbalance, GridSearchCV for tuning, SHAP for explainability.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.data_preprocessing import preprocess_pipeline

print("All libraries loaded ✓")

## 1. Data Preprocessing

In [ ]:
# Run preprocessing pipeline (clean → engineer features → encode → split → scale)
X_train, X_test, y_train, y_test, scaler, feature_names = preprocess_pipeline(
    "../data/raw data/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "../data/processed data/processed_data.csv"
)

print(f"\nClass distribution (train):")
print(y_train.value_counts().rename({0: "No Churn", 1: "Churn"}))

## 2. Handle Class Imbalance with SMOTE

The dataset is imbalanced (~27% churn). SMOTE creates synthetic minority samples to balance the training set.

In [ ]:
# Apply SMOTE to training data ONLY (never to test data)
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {len(y_train)} samples")
print(f"  No Churn: {(y_train == 0).sum()} | Churn: {(y_train == 1).sum()}")
print(f"\nAfter SMOTE:  {len(y_train_sm)} samples")
print(f"  No Churn: {(y_train_sm == 0).sum()} | Churn: {(y_train_sm == 1).sum()}")

## 3. Train Models

In [ ]:
# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                              random_state=42, eval_metric="logloss"),
}

# Train and evaluate each model
results = []
trained_models = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"  Training: {name}")
    print(f"{'='*50}")
    
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }
    results.append(metrics)
    trained_models[name] = model
    
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

results_df = pd.DataFrame(results).set_index("Model")
print("\n" + "="*60)
print("  MODEL COMPARISON")
print("="*60)
display(results_df.round(4))

## 4. Model Evaluation — ROC Curves & Confusion Matrices

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {"Logistic Regression": "#3498db", "Random Forest": "#2ecc71", "XGBoost": "#e74c3c"}

# ROC Curve plot
ax = axes[0]
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=colors[name], linewidth=2)

ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Model Comparison", fontweight="bold")
ax.legend(loc="lower right")

# Metrics comparison bar chart
ax2 = axes[1]
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
x = np.arange(len(metrics_to_plot))
width = 0.25

for i, (name, row) in enumerate(results_df.iterrows()):
    ax2.bar(x + i * width, [row[m] for m in metrics_to_plot], width,
            label=name, color=list(colors.values())[i], edgecolor="black")

ax2.set_xticks(x + width)
ax2.set_xticklabels(metrics_to_plot)
ax2.set_ylim(0, 1.1)
ax2.set_title("Metrics Comparison", fontweight="bold")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["No Churn", "Churn"],
        cmap="Blues", ax=axes[i]
    )
    axes[i].set_title(f"{name}", fontweight="bold")

plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning (Best Model)

Using GridSearchCV with 5-fold cross-validation on the best-performing model.

In [ ]:
# Identify best model from initial comparison
best_model_name = results_df["ROC-AUC"].idxmax()
print(f"Best model: {best_model_name} — tuning hyperparameters...\n")

# GridSearchCV for XGBoost
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1, 0.2],
}

grid_search = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_sm, y_train_sm)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

# Evaluate tuned model on test set
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

print(f"\n── Tuned Model on Test Set ──")
print(classification_report(y_test, y_pred_tuned, target_names=["No Churn", "Churn"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_tuned):.4f}")

## 6. Feature Importance & SHAP Explainability

Understanding *why* the model predicts churn — critical for business trust and actionable insights.

In [ ]:
# Feature importance from the tuned XGBoost model
importances = best_model.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.tail(15).plot(kind="barh", ax=ax, color="#3498db", edgecolor="black")
ax.set_title("Top 15 Feature Importances (XGBoost)", fontweight="bold", fontsize=14)
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Explainability
import shap

# Create SHAP explainer
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# Summary plot — global feature importance via SHAP
print("SHAP Summary Plot — shows impact of each feature on prediction:")
shap.summary_plot(shap_values, X_test, feature_names=X_train.columns, show=True)

In [ ]:
# SHAP bar plot — mean absolute SHAP values
shap.summary_plot(shap_values, X_test, feature_names=X_train.columns, plot_type="bar", show=True)

## 7. Save the Best Model

In [ ]:
# Save the tuned model bundle (model + scaler + feature names)
models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)

bundle = {
    "model": best_model,
    "scaler": scaler,
    "feature_names": feature_names,
    "model_name": "XGBoost (Tuned)",
}

save_path = os.path.join(models_dir, "churn_model.pkl")
joblib.dump(bundle, save_path)
print(f"Model saved → {save_path}")

# Verify: load and predict
loaded = joblib.load(save_path)
test_pred = loaded["model"].predict(X_test[:1])
print(f"Verification — prediction on first test sample: {'Churn' if test_pred[0] == 1 else 'No Churn'} ✓")

## Summary

- **3 models trained**: Logistic Regression (baseline), Random Forest, XGBoost
- **SMOTE** balanced the training data from ~27% churn to 50/50
- **XGBoost** performed best with highest ROC-AUC
- **GridSearchCV** tuned hyperparameters for optimal performance
- **SHAP** revealed that tenure, contract type, and monthly charges are the top churn drivers
- Model bundle (model + scaler + features) saved to `models/churn_model.pkl`

**Next step**: Deploy via Streamlit dashboard → `streamlit run app.py`